# Liver Fibrosis — Colab 訓練

在 Colab 跑 `train.py`。程式碼本身不用改:`get_device()` 會自動選 CUDA,
`DATA_DIR` 用環境變數覆蓋。

## ⚠️ 一個會讓你等很久的坑

`config.DEDUP=True` 會對每個檔案算內容 hash(這是本專案最重要的一步,見 `dataset.py` 的
LEAKAGE 警告)。6323 個檔案在**本地磁碟**約 5 秒,但如果 `DATA_DIR` 直接指到
**掛載的 Google Drive**,6323 次檔案開啟走 FUSE 會慢到幾分鐘起跳,而且每次重跑都要再等一次。

→ **一定要把資料解壓/複製到 `/content`,不要直接讀 Drive。**

## 執行順序
由上而下跑一次即可。第 3 節(取得資料)有三種來源,擇一。

## 1. 確認 GPU

沒有 GPU 的話:「執行階段 → 變更執行階段類型 → T4 GPU」。
CPU 也跑得動(資料量小),只是會慢很多。

In [ ]:
!nvidia-smi || echo '沒有 GPU,會退回 CPU'
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. 取得程式碼

In [ ]:
!git clone https://github.com/tony921023/liver-fibrosis-test.git /content/test
%cd /content/test

# Colab 已預裝 torch/torchvision/sklearn/matplotlib/pandas/numpy,
# 這行實際上只會補裝 kaggle。requirements.txt 用寬鬆版本,不會把 Colab 的 torch 降級。
!pip install -q -r requirements.txt

## 3. 取得資料(三選一)

`data/` 在 `.gitignore` 裡,clone 下來是空的,要另外抓。
**三種方式的共同目標:讓 F0~F4 五個資料夾出現在 `/content` 底下的某處。**

### 3a. 從 Kaggle 下載(推薦)

In [ ]:
from google.colab import files
files.upload()          # 上傳 kaggle.json

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# TODO: 換成實際的 dataset slug
!kaggle datasets download -d <owner>/<dataset-name> -p /content/dl --unzip
!find /content/dl -maxdepth 3 -type d | head -20

### 3b. 從 Google Drive 複製

注意是**複製到 `/content`**,不是直接拿 Drive 路徑當 `DATA_DIR`(見開頭的坑)。
如果 Drive 上放的是 zip,解壓到 `/content` 會比複製幾千個小檔快得多。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 情況一:Drive 上是 zip(快)
!mkdir -p /content/dl && unzip -q '/content/drive/MyDrive/<路徑>/dataset.zip' -d /content/dl

# 情況二:Drive 上是解開的資料夾(慢,幾千個小檔)
# !cp -r '/content/drive/MyDrive/<路徑>/Dataset' /content/dl/

!find /content/dl -maxdepth 3 -type d | head -20

### 3c. 直接上傳 zip

In [ ]:
from google.colab import files
up = files.upload()     # 選你的 dataset zip

import pathlib
zname = next(iter(up))
!mkdir -p /content/dl && unzip -q "{zname}" -d /content/dl
!find /content/dl -maxdepth 3 -type d | head -20

## 4. 自動找出 F0~F4 的所在目錄,設成 DATA_DIR

不同來源解壓出來的巢狀層數常常不一樣,所以用搜尋的,不要手寫死路徑。

`os.environ` 的改動會被 `!python train.py` 這種 shell 子行程繼承,所以設一次就好,
不用改 `config.py`。

In [ ]:
import os, pathlib

root = None
for p in pathlib.Path('/content').rglob('F0'):
    if 'drive' in p.parts:            # 跳過 Drive 掛載點,只找 /content 本地磁碟
        continue
    if p.is_dir() and all((p.parent / f'F{i}').is_dir() for i in range(5)):
        root = p.parent
        break

assert root is not None, '找不到同時含 F0~F4 的目錄,請檢查第 3 節解壓結果'
os.environ['DATA_DIR'] = str(root)
print('DATA_DIR =', root)

total = 0
for i in range(5):
    n = sum(1 for _ in (root / f'F{i}').iterdir())
    total += n
    print(f'  F{i}: {n} files')
print(f'  total: {total}   (本機那份 = 6323)')

## 5. 資料健檢:確認 Kaggle 這份跟本機那份是同一個東西

本機那份的事實(見 `CLAUDE.md`):

| 項目 | 期望值 |
|---|---|
| 檔案數 | 6323 |
| **不重複影像數** | **1536** |
| 跨類別重複的 hash | 0(標籤沒有矛盾) |

**如果 unique 數對不上,先搞清楚為什麼再訓練** —— 表示 Kaggle 版本跟本機這份不同,
後面所有數字都不能跟舊 run 比較。

In [ ]:
import hashlib, collections, time, pathlib, os

root = pathlib.Path(os.environ['DATA_DIR'])
t = time.time()
hash_to_classes = collections.defaultdict(set)
n = 0
for i in range(5):
    for f in sorted((root / f'F{i}').iterdir()):
        n += 1
        hash_to_classes[hashlib.md5(f.read_bytes()).hexdigest()].add(i)

cross = sum(1 for v in hash_to_classes.values() if len(v) > 1)
print(f'耗時 {time.time() - t:.1f}s   (本地磁碟應該 <15s;若破分鐘,代表你在讀 Drive)')
print(f'檔案數        : {n}              (期望 6323)')
print(f'不重複影像數  : {len(hash_to_classes)}   (期望 1536)')
print(f'跨類別重複    : {cross}              (期望 0)')

## 6. 訓練

Colab 通常有 4~8 個 vCPU,`NUM_WORKERS` 可以比 Mac 上調大。

resnet18 + 1074 張訓練圖 + 30 epochs,在 T4 上大約幾分鐘。

⚠️ 預期 test macro AUROC 大概落在 **0.75~0.90**。
如果看到 0.99,先檢查 `config.DEDUP` 是不是變成 `False` 了 —— 那個數字是模型在背圖,不是在判讀。

In [ ]:
!sed -i 's/^NUM_WORKERS = .*/NUM_WORKERS = 4                # Colab: 比 Mac 調大/' config.py
!grep -E '^(DEDUP|BACKBONE|EPOCHS|NUM_WORKERS|TTA|AUG_STRENGTH) ' config.py

!python train.py

## 7. 對照實驗:換 backbone

跑完 baseline 之後,換大一點的 backbone 再跑一次比較。
可選:`resnet18` / `resnet34` / `resnet50` / `resnet101` / `efficientnet_b0` / `convnext_tiny`

每次 `train.py` 都會覆寫 `outputs/`,所以下面每跑一輪就把結果另存一份。

In [ ]:
import json, shutil, pathlib, subprocess

results = {}
for bb in ['resnet18', 'resnet50', 'convnext_tiny']:
    print(f'\n{"=" * 60}\n  {bb}\n{"=" * 60}')
    subprocess.run(['sed', '-i', f's/^BACKBONE = .*/BACKBONE = "{bb}"/', 'config.py'], check=True)
    subprocess.run(['python', 'train.py'], check=True)

    rep = json.loads(pathlib.Path('outputs/test_report.json').read_text())
    results[bb] = rep
    shutil.copytree('outputs', f'outputs_{bb}', dirs_exist_ok=True)

print(f'\n{"backbone":<18}{"macro AUROC":>13}{"bal. acc":>11}{"QWK":>9}')
for bb, r in results.items():
    print(f'{bb:<18}{r["macro_auroc"]:>13.4f}{r["balanced_accuracy"]:>11.4f}'
          f'{r["quadratic_weighted_kappa"]:>9.4f}')

## 8. 把結果抓回本機

Colab 執行階段結束後 `/content` 會清空,想留的東西記得下載或存回 Drive。

`checkpoints/best.pt` 有幾十 MB,視需要再抓。

In [ ]:
!zip -qr /content/results.zip outputs* checkpoints

from google.colab import files
files.download('/content/results.zip')

# 或存回 Drive(需先在 3b 掛載):
# !cp /content/results.zip '/content/drive/MyDrive/'